In [ ]:
import json
import os
import time
import sys

import dask.dataframe as dd
import dask_geopandas as dg
import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
import contextily as cx

import matplotlib.cm as cm
import matplotlib.colors as colors

from matplotlib.patches import Patch

sys.path.insert(1, '../0_prep_NHD/')
from reaches import *
from utils import *

### Functions

In [ ]:
def getFiles(filepath):
    files = []
    for file in os.listdir(filepath):
        if not file.startswith('.'):
            # if not 'C2' in file:
            files.append(os.path.join(filepath, file))
    return files

In [ ]:
def getNHDFiles(filedir, huc2s):
    files = []
    for huc2 in huc2s:
        filepath = f'{filedir}{huc2}'
        for file in os.listdir(filepath):
            if not file.startswith('.'):
                files.append(os.path.join(filepath, file))
    return files

### Setup

In [ ]:
# Order of all possible bins for plotting
order = ['(0, 10]', '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]']

In [ ]:
save_dir = f'/nas/cee-water/cjgleason/fiona/for_presentations/2025_SWOT_ST/results/'

In [ ]:
huc2s = ['01', '02', '03', '04', '05', '06', '07', '08', '09',
         '10', '11', '12', '13', '14', '15', '16', '17', '18']

### Get all gauge IDs

In [ ]:
nhd_dir = f'/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_data/NHD_prepped/HUC2_'

In [ ]:
# Get list of NHDfiles
nhd_files = getNHDFiles(nhd_dir, huc2s)
print(len(nhd_files))

In [ ]:
# Read in all files
nhd = dg.read_parquet(nhd_files,
                      columns=['NHDPlusID', 'GNIS_Name', 'LengthKM',
                               'WidthM', 'Bin', 'StreamOrde',
                               'Slope', 'Sinuosity', 'GageID', 'geometry']
                     )

In [ ]:
nhd = nhd[~nhd['GageID'].isna()]
nhd = nhd.drop_duplicates(subset='NHDPlusID')

nhd = nhd.compute()

In [ ]:
nhd['Bin'] = pd.Categorical(nhd['Bin'], categories=order, ordered=True)

In [ ]:
nhd[nhd['LengthKM'] > 20]

In [ ]:
gauge_ids = nhd['GageID'].to_list()

In [ ]:
len(gauge_ids)

In [ ]:
outpath = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data'

In [ ]:
with open(os.path.join(outpath, 'nhd_gauges.json'), 'w', encoding='utf-8') as f:
                json.dump(gauge_ids, f)

### Read results

In [ ]:
gauge_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/data/usgs_gauges.json'

In [ ]:
with open(gauge_path, 'r') as file:
        gauges = json.load(file)

In [ ]:
gauges.index('01171300')